In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

c:\Users\Chinmay\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_DIR = "./sms_model"              # folder you showed in the screenshot
ONNX_PATH = "./sms_classifier.onnx"    # output file
MAX_LENGTH = 128

In [3]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

Loading tokenizer...


In [4]:
print("Loading model...")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.eval()

Loading model...


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [5]:
dummy_text = "Your A/C XX1234 is debited for Rs. 500 via UPI"

inputs = tokenizer(
    dummy_text,
    return_tensors="pt",
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH,
)

input_ids = inputs["input_ids"]
attention_mask = inputs["attention_mask"]

print("Input shape:", input_ids.shape)

Input shape: torch.Size([1, 128])


In [7]:
pip install onnxscript

   ---------------------------------------- 0.0/693.4 kB ? eta -:--:--
   ---------------------------------------- 693.4/693.4 kB 7.6 MB/s  0:00:00
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   -- ------------------------------------- 1.0/16.4 MB 4.3 MB/s eta 0:00:04
   -- ------------------------------------- 1.0/16.4 MB 4.3 MB/s eta 0:00:04
   -- ------------------------------------- 1.0/16.4 MB 4.3 MB/s eta 0:00:04
   ---- ----------------------------------- 1.8/16.4 MB 2.1 MB/s eta 0:00:08
   ------ --------------------------------- 2.6/16.4 MB 2.3 MB/s eta 0:00:06
   -------- ------------------------------- 3.4/16.4 MB 2.5 MB/s eta 0:00:06
   ---------- ----------------------------- 4.2/16.4 MB 2.8 MB/s eta 0:00:05
   ---------- ----------------------------- 4.2/16.4 MB 2.8 MB/s eta 0:00:05
   ---------- ----------------------------- 4.2/16.4 MB 2.8 MB/s eta 0:00:05
   ---------- ----------------------------- 4.2/16.4 MB 2.8 MB/s eta 0:00:05
   ---------

In [8]:
print("Exporting to ONNX...")

torch.onnx.export(
    model,
    (input_ids, attention_mask),
    ONNX_PATH,
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch_size"},
        "attention_mask": {0: "batch_size"},
        "logits": {0: "batch_size"},
    },
    opset_version=13,
    do_constant_folding=True,
)

print("✅ ONNX model exported successfully to:", ONNX_PATH)

Exporting to ONNX...


C:\Users\Chinmay\AppData\Local\Temp\ipykernel_35740\2187353890.py:3: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0115 12:30:33.470000 35740 site-packages\torch\onnx\_internal\exporter\_compat.py:137] Setting ONNX exporter to use operator set version 18 because the requested opset_version 13 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0115 12:30:33.954000 35740 site-packages\torch\onnx\_internal\exporter\_registration.py:110] torchvision is not installed. Skipping torchvision::nms
W0115 12:30:34.083000 35740 

[torch.onnx] Obtain model graph for `DistilBertForSequenceClassification([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `DistilBertForSequenceClassification([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


c:\Users\Chinmay\miniconda3\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 13).
Failed to convert the model to the target version 13 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "c:\Users\Chinmay\miniconda3\Lib\site-packages\onnxscript\version_converter\__init__.py", line 127, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "c:\Users\Chinmay\miniconda3\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "c:\Users\Chinmay\miniconda3\Lib\site-packages\onnxscript\version_converter\__init__.py", li

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 28 of general pattern rewrite rules.
✅ ONNX model exported successfully to: ./sms_classifier.onnx


In [9]:
pip install onnx onnxruntime

   ---------------------------------------- 0.0/13.5 MB ? eta -:--:--
   -- ------------------------------------- 0.8/13.5 MB 7.1 MB/s eta 0:00:02
   ----- ---------------------------------- 1.8/13.5 MB 4.6 MB/s eta 0:00:03
   ------ --------------------------------- 2.1/13.5 MB 3.6 MB/s eta 0:00:04
   ---------- ----------------------------- 3.4/13.5 MB 4.2 MB/s eta 0:00:03
   ------------- -------------------------- 4.5/13.5 MB 4.1 MB/s eta 0:00:03
   --------------- ------------------------ 5.2/13.5 MB 4.1 MB/s eta 0:00:03
   ----------------- ---------------------- 6.0/13.5 MB 4.1 MB/s eta 0:00:02
   -------------------- ------------------- 6.8/13.5 MB 4.0 MB/s eta 0:00:02
   ---------------------- ----------------- 7.6/13.5 MB 4.0 MB/s eta 0:00:02
   ------------------------ --------------- 8.4/13.5 MB 4.0 MB/s eta 0:00:02
   ---------------------------- ----------- 9.4/13.5 MB 4.0 MB/s eta 0:00:02
   ------------------------------ --------- 10.2/13.5 MB 4.0 MB/s eta 0:00:01
   --

In [10]:
import onnx
import onnxruntime as ort

# Load and validate ONNX model
onnx_model = onnx.load("sms_classifier.onnx")
onnx.checker.check_model(onnx_model)

# Create inference session
sess = ort.InferenceSession("sms_classifier.onnx")

print("ONNX Inputs:", [i.name for i in sess.get_inputs()])
print("ONNX Outputs:", [o.name for o in sess.get_outputs()])

ONNX Inputs: ['input_ids', 'attention_mask']
ONNX Outputs: ['logits']
